In [0]:
Raw CSV
   ↓
Ingest
   ↓
Transform
   ↓
Load
   ↓
Analytics-ready data

In [0]:
Extract
  ↓
Read raw CSV
  ↓
Transform
  ↓
Clean + calculate
  ↓
Load
  ↓
Delta table

In [0]:
                RAW DATA
              sales.csv
                  │
                  ▼
              EXTRACT
                  │
                  ▼
             PYSPARK
                  │
        ┌─────────┴─────────┐
        ▼                   ▼
     CLEAN               TRANSFORM
        │                   │
        │            Calculate revenue
        │            Standardize data
        │            Handle nulls
        └─────────┬─────────┘
                  ▼
              LOAD
                  │
                  ▼
            DELTA TABLE
                  │
                  ▼
            SQL ANALYSIS

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RetailSalesETL").getOrCreate()

df_raw = spark.read.csv(
    "/Workspace/Users/sauichyansalai@gmail.com/sales.csv",
    header=True,
    inferSchema=True
)

df_raw.show()

+--------+----------+-------------+--------+-----------+--------+-----+
|order_id|order_date|customer_name| product|   category|quantity|price|
+--------+----------+-------------+--------+-----------+--------+-----+
|    1001|2026-01-05|         Ravi|  Laptop|Electronics|       2|50000|
|    1002|2026-01-06|          Anu|   Mouse|Electronics|       3|  500|
|    1003|2026-01-07|        Kumar|Keyboard|Electronics|       2| 1200|
|    1004|2026-01-08|        Priya|   Chair|  Furniture|       1| 4500|
|    1005|2026-01-09|         Arun|  Laptop|Electronics|       1|50000|
|    1006|2026-01-10|        Divya|    Desk|  Furniture|       2| 7000|
|    1007|2026-01-11|         Ravi|   Mouse|Electronics|    NULL|  500|
|    1008|2026-01-12|          Anu|Keyboard|Electronics|       1| 1200|
|    1009|2026-01-13|        Kumar|   Chair|  Furniture|       2| 4500|
|    1010|2026-01-14|        Priya|  Laptop|Electronics|       1|50000|
+--------+----------+-------------+--------+-----------+--------

In [0]:
df_raw.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: integer (nullable = true)



In [0]:
df_raw.show(10)

+--------+----------+-------------+--------+-----------+--------+-----+
|order_id|order_date|customer_name| product|   category|quantity|price|
+--------+----------+-------------+--------+-----------+--------+-----+
|    1001|2026-01-05|         Ravi|  Laptop|Electronics|       2|50000|
|    1002|2026-01-06|          Anu|   Mouse|Electronics|       3|  500|
|    1003|2026-01-07|        Kumar|Keyboard|Electronics|       2| 1200|
|    1004|2026-01-08|        Priya|   Chair|  Furniture|       1| 4500|
|    1005|2026-01-09|         Arun|  Laptop|Electronics|       1|50000|
|    1006|2026-01-10|        Divya|    Desk|  Furniture|       2| 7000|
|    1007|2026-01-11|         Ravi|   Mouse|Electronics|    NULL|  500|
|    1008|2026-01-12|          Anu|Keyboard|Electronics|       1| 1200|
|    1009|2026-01-13|        Kumar|   Chair|  Furniture|       2| 4500|
|    1010|2026-01-14|        Priya|  Laptop|Electronics|       1|50000|
+--------+----------+-------------+--------+-----------+--------

In [0]:
from pyspark.sql.functions import col, coalesce, lit

df_clean = df_raw.withColumn(
    "quantity",
    coalesce(col("quantity"), lit(1))
)

In [0]:
from pyspark.sql.functions import initcap

df_clean = df_clean.withColumn(
    "product",
    initcap(col("product"))
)

In [0]:
df_transformed = df_clean.withColumn(
    "total_amount",
    col("quantity") * col("price")
)

In [0]:
df_final = df_transformed.select(
    "order_id",
    "order_date",
    "customer_name",
    "product",
    "category",
    "quantity",
    "price",
    "total_amount"
)

In [0]:
df_final.show()

+--------+----------+-------------+--------+-----------+--------+-----+------------+
|order_id|order_date|customer_name| product|   category|quantity|price|total_amount|
+--------+----------+-------------+--------+-----------+--------+-----+------------+
|    1001|2026-01-05|         Ravi|  Laptop|Electronics|       2|50000|      100000|
|    1002|2026-01-06|          Anu|   Mouse|Electronics|       3|  500|        1500|
|    1003|2026-01-07|        Kumar|Keyboard|Electronics|       2| 1200|        2400|
|    1004|2026-01-08|        Priya|   Chair|  Furniture|       1| 4500|        4500|
|    1005|2026-01-09|         Arun|  Laptop|Electronics|       1|50000|       50000|
|    1006|2026-01-10|        Divya|    Desk|  Furniture|       2| 7000|       14000|
|    1007|2026-01-11|         Ravi|   Mouse|Electronics|       1|  500|         500|
|    1008|2026-01-12|          Anu|Keyboard|Electronics|       1| 1200|        1200|
|    1009|2026-01-13|        Kumar|   Chair|  Furniture|       2|

In [0]:
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_sales")

Python / PySpark
       ↓
Transform
       ↓
Delta
       ↓
retail_sales

In [0]:
%sql
SELECT *
FROM retail_sales;

order_id,order_date,customer_name,product,category,quantity,price,total_amount
1001,2026-01-05,Ravi,Laptop,Electronics,2,50000,100000
1002,2026-01-06,Anu,Mouse,Electronics,3,500,1500
1003,2026-01-07,Kumar,Keyboard,Electronics,2,1200,2400
1004,2026-01-08,Priya,Chair,Furniture,1,4500,4500
1005,2026-01-09,Arun,Laptop,Electronics,1,50000,50000
1006,2026-01-10,Divya,Desk,Furniture,2,7000,14000
1007,2026-01-11,Ravi,Mouse,Electronics,1,500,500
1008,2026-01-12,Anu,Keyboard,Electronics,1,1200,1200
1009,2026-01-13,Kumar,Chair,Furniture,2,4500,9000
1010,2026-01-14,Priya,Laptop,Electronics,1,50000,50000


In [0]:
%sql
SELECT
    SUM(total_amount) AS total_revenue
FROM retail_sales;

total_revenue
233100


Revenue By Catagory

In [0]:
%sql
SELECT
    category,
    SUM(total_amount) AS revenue
FROM retail_sales
GROUP BY category
ORDER BY revenue DESC;

category,revenue
Electronics,205600
Furniture,27500


In [0]:
%sql
select product, sum(total_amount) as revenue
from retail_sales
group by product
order by revenue desc

product,revenue
Laptop,200000
Desk,14000
Chair,13500
Keyboard,3600
Mouse,2000


In [0]:
retail-sales-etl-pipeline/
│
├── data/
│   └── sales.csv
│
├── notebooks/
│   └── retail_sales_etl.py
│
├── sql/
│   └── analysis_queries.sql
│
├── README.md
│
└── architecture.png